In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

def engineer_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    df['Temperature difference'] = (df['Process temperature [K]'] - df['Air temperature [K]']).round(1)

    df['Power [W]'] = (df['Rotational speed [rpm]'] * df['Torque [Nm]'] * (2 * np.pi / 60)).round(2)

    df['OSF criterion'] = df['Tool wear [min]'] * df['Torque [Nm]']

    df = pd.get_dummies(df, columns=['Type'], drop_first=True, dtype=int)

    return df

engineer_features(pd.read_csv("../data/test/valid_input.csv"))

def run_batch(input_path: str | Path, output_path: str | Path = None, model_path="../models/final_model.joblib"):
    artifact = joblib.load(model_path)
    model = artifact['model']
    scaler = artifact['scaler']
    features = artifact['features']
    numeric_features = artifact['numeric_features']

    df_raw = pd.read_csv(input_path)
    df = engineer_features(df_raw)
    df = df[features]

    df_scaled = df.copy()
    df_scaled[numeric_features] = scaler.transform(df[numeric_features])

    proba = model.predict_proba(df_scaled)[:, 1]
    result = pd.DataFrame({
        "record_index": df_raw.index,
        "probability": proba,
    })

    if output_path is not None:
        result.to_csv(output_path, index=False)

    return result

print(run_batch("../data/test/valid_input.csv",))

   record_index  probability
0             0     0.000680
1             1     0.002840
2             2     0.084365
